# 05 第三章实验框架

本 Notebook 建立后续实验入口。第一轮重点是让每个方向都有可运行示例，后续可以逐步扩展成专题实验。

当前包含：

* Fourier 逼近周期函数；
* Gibbs 现象的最小示例；
* 自适应分段线性逼近；
* 后续实验目录。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import adaptive_piecewise_linear, piecewise_linear_interpolate


## Fourier 逼近周期函数

三角多项式逼近写作

$$
S_N(x)=a_0+\sum_{k=1}^{N}\left(a_k\cos kx+b_k\sin kx\right).
$$

离散 Fourier 变换把等距采样值转换为频率系数。这里用 `numpy.fft` 做一个最小示例，重点是说明 FFT 服务于三角多项式逼近。


In [ ]:
N = 128
x = np.linspace(0.0, 2 * np.pi, N, endpoint=False)
y = np.sin(3 * x) + 0.4 * np.cos(5 * x)
coefficients = np.fft.rfft(y) / N
frequencies = np.fft.rfftfreq(N, d=1 / N)

plt.figure(figsize=(8, 4))
plt.stem(frequencies[:12], np.abs(coefficients[:12]))
plt.title("周期函数的离散频谱")
plt.xlabel("频率编号")
plt.ylabel("系数幅值")
plt.tight_layout()
plt.show()


## Gibbs 现象入口

对不连续周期函数，Fourier 部分和在跳跃点附近会出现过冲。增加项数会压缩振荡区域，但不会简单消除过冲高度。下面只给出可运行入口，完整推导后续补充。


In [ ]:
def square_wave(x):
    return np.where(np.sin(x) >= 0, 1.0, -1.0)

x_grid = np.linspace(-np.pi, np.pi, 1024, endpoint=False)
y_square = square_wave(x_grid)
fft_coefficients = np.fft.fft(y_square)

plt.figure(figsize=(8, 4.5))
plt.plot(x_grid, y_square, color="gray", label="方波")
for keep in [5, 15, 45]:
    filtered = np.zeros_like(fft_coefficients)
    filtered[: keep + 1] = fft_coefficients[: keep + 1]
    filtered[-keep:] = fft_coefficients[-keep:]
    y_partial = np.fft.ifft(filtered).real
    plt.plot(x_grid, y_partial, label=f"保留 {keep} 个频率")
plt.title("Gibbs 现象入口示例")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()


## 自适应分段线性逼近

自适应逼近的思想是：在误差大的地方加密节点，在误差小的地方减少节点。下面先给出中点误差估计的分段线性版本。


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_adapt, y_adapt = adaptive_piecewise_linear(runge, -1.0, 1.0, tolerance=2e-3, max_depth=12)
x_eval = np.linspace(-1.0, 1.0, 600)
y_eval = piecewise_linear_interpolate(x_adapt, y_adapt, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, runge(x_eval), label="目标函数")
plt.plot(x_eval, y_eval, "--", label="自适应分段线性逼近")
plt.scatter(x_adapt, y_adapt, s=18, color="black", label="自适应节点")
plt.title("自适应节点在曲率较大区域加密")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()

print("自适应节点数:", x_adapt.size)


## 后续实验目录

后续可以把以下内容补成独立专题：

* Chebyshev 级数逼近不同光滑性函数；
* Legendre 投影与 Chebyshev 拟合误差对比；
* 正交多项式最小二乘拟合与幂基拟合对比；
* Taylor 与 Padé 对比更多函数，例如 $\log(1+x)$ 和 $\tan x$；
* Gibbs 现象的过冲高度与项数关系；
* 自适应三次样条逼近；
* 正则化最小二乘拟合。


## 实验小结

本 Notebook 目前是实验框架，不追求一次性讲完所有理论。它的作用是为后续扩展保留可运行入口，并展示第三章的主要实验方向。
